# Differentiable FEA + PINN Journey

This notebook drills into Approach A: data generation via differentiable FEA, surrogate training, validation, and generalization outcomes.



## Story Outline

1. **Imports & Config**  
   - Load numpy/pandas/torch/plotly.  
   - Path helpers pointing at `src/approach_a_pinn` & `data/results`.
2. **Geometry + Load Recap**  
   - Inline call to shared drawing helpers (reuse from master).  
   - Connect each loadcase to dataset columns.
3. **Data Generation Walkthrough**  
   - Summarize `GenerateData_Lshape_Batch.jl` + scenario rollouts.  
   - Visualize coverage from `multi_geom_training/*.csv` with histograms + scatter overlays.  
   - Provide toggles to switch between legacy vs multi-geometry corpora.
4. **Model Architecture & Training**  
   - Auto-load `TrainPINN.py` hyperparameters, render network diagram.  
   - Reconstruct training curves directly from log files (CSV).  
   - Show inference timing snippet replicating `Proof_Speedup` but referencing multi-geometry checkpoint.
5. **Validation & Generalization**  
   - Load `pinn_generalization_test.csv` + `multi_geom_model_metrics.csv`, plot MAE/MAPE bars.  
   - 3D compliance surface (Plotly) for selected loadcases predicted vs ground truth.
6. **Integration Hooks**  
   - Provide function that master & MMC notebooks can call to get surrogate predictions under specific loadcases.  
   - Document commands to retrain & refresh artifacts.



## Key Terminology for PINN Approach

- **Surrogate Model**: Neural network that approximates expensive FEA computations. Trained once, then used for fast predictions.
- **Training Data**: Pairs of (screw positions, compliance values) from FEA simulations. The model learns the mapping.
- **Generalization**: Model's ability to predict accurately on new screw positions/geometries not seen during training.
- **MAE (Mean Absolute Error)**: Average prediction error in Joules. Lower = more accurate.
- **MAPE (Mean Absolute Percentage Error)**: Error as percentage. 5% MAPE means predictions are within 5% of truth.
- **R² (R-squared)**: How well model explains variance. 1.0 = perfect, 0.0 = no better than predicting the mean.
- **One-Hot Encoding**: Categorical variables (geometry type, load case) converted to binary vectors for neural network input.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn

_possible_dirs = [Path.cwd().resolve()]
if (Path.cwd() / 'docs' / 'notebooks').exists():
    _possible_dirs.append(Path.cwd() / 'docs' / 'notebooks')
helper_dir = None
for candidate in _possible_dirs:
    candidate = candidate.resolve()
    if (candidate / 'story_helpers.py').exists():
        helper_dir = candidate
        break
if helper_dir is None:
    raise RuntimeError('Cannot locate story_helpers.py')
if str(helper_dir) not in sys.path:
    sys.path.append(str(helper_dir))

from story_helpers import (
    ROOT,
    RESULTS_DIR,
    PINN_DIR,
    assemble_multi_geom_dataframe,
    draw_lbracket_2d,
    list_multi_geom_paths,
    make_bracket_mesh,
    plot_dataset_histogram,
    plot_dataset_scatter,
    summarize_dataset,
    SurrogateModel,
)

# Configure Plotly for VS Code Jupyter extension
try:
    pio.renderers.default = "vscode"
except ValueError:
    try:
        pio.renderers.default = "notebook"
    except ValueError:
        # Fallback: let Plotly auto-detect
        pass

SCREW_COLS = ["s1_x", "s1_y", "s2_x", "s2_y", "s3_x", "s3_y"]


## Geometry & Load Recap

We start by reusing the shared geometry helpers to visualize the L-bracket and its two canonical load cases. This mirrors the master story but keeps the context close to the PINN workflow.



In [2]:
draw_lbracket_2d().show()
make_bracket_mesh().show()


**What this shows:**
- **2D Planform**: L-bracket outline with void region and fixed support
- **3D Geometry**: Extruded view showing 5mm thickness
- **All Geometries**: Ribbed channel and tapered plate 3D meshes
- **Load Cases**: Example showing how force vectors are applied in 3D space
- **Purpose**: Establishes the physical problem that the PINN will learn to solve

## Multi-Geometry Dataset Walkthrough

We materialize the dataset manifest directly from the CSV corpus in `data/results/multi_geom_training`. This replaces the historical histogram PNGs with executable Plotly figures.



In [3]:
def load_multi_dataset() -> pd.DataFrame:
    frames = []
    for path in list_multi_geom_paths():
        df = pd.read_csv(path)
        for col in SCREW_COLS:
            if col not in df.columns:
                df[col] = 0.0
        frames.append(df)
    if not frames:
        raise RuntimeError("No CSV files found in data/results/multi_geom_training")
    df = pd.concat(frames, ignore_index=True)
    return df

multi_df = load_multi_dataset()
print(f"Loaded {len(multi_df)} samples across {multi_df['geometry'].nunique()} geometries and {multi_df['load_case'].nunique()} load cases.")

agg = multi_df.groupby(["geometry", "load_case"], as_index=False).agg(
    samples=("geometry", "count"),
    compliance_min=("compliance", "min"),
    compliance_max=("compliance", "max"),
    compliance_mean=("compliance", "mean"),
)
display(agg)



Loaded 1020 samples across 3 geometries and 5 load cases.


,geometry,load_case,samples,compliance_min,compliance_max,compliance_mean
0,l_bracket,horizontal_tip,200,205.316302,479.239144,378.568877
1,l_bracket,vertical_tip,200,972.439491,1701.221319,1247.291741
2,ribbed_channel,lateral_shear,220,0.619838,0.856086,0.741916
3,ribbed_channel,upward_tip,220,32.139060,32.438080,32.290524
4,tapered_plate,combined_tip,180,22.419019,23.574407,22.897830


**What this shows:**
- **Sample Counts**: Number of training samples per geometry/load case combination
- **Compliance Statistics**: Min, max, and mean compliance values for each scenario
- **Dataset Coverage**: Shows which geometries and load cases are represented and how many samples each has
- **Purpose**: Documents the training dataset composition and helps identify any imbalances or gaps in coverage

In [4]:
sample_bar = px.bar(
    agg,
    x="geometry",
    y="samples",
    color="load_case",
    title="Sample Count per Geometry & Load Case",
    barmode="stack",
)
sample_bar.update_layout(xaxis_title="Geometry", yaxis_title="Number of Samples")
sample_bar.show()

hist_fig = px.histogram(
    multi_df,
    x="compliance",
    color="geometry",
    nbins=30,
    opacity=0.7,
    title="Compliance Distribution Across Geometries",
)
hist_fig.update_layout(xaxis_title="Compliance (J)")
hist_fig.show()



**What this shows:**
- **Stacked Bar Chart**: Visual breakdown of sample counts by geometry and load case
- **Color Coding**: Each color represents a different load case
- **Purpose**: Quickly see which geometry/load combinations have the most training data (important for understanding model generalization)

In [5]:
legacy_path = RESULTS_DIR / "pinn_training_data.csv"
legacy_hist = plot_dataset_histogram(legacy_path, "Legacy Single-Geometry Dataset")
legacy_hist.show()
legacy_scatter = plot_dataset_scatter(legacy_path, "Legacy Single-Geometry Dataset")
legacy_scatter.show()



In [6]:
scatter_fig = plot_dataset_scatter(
    RESULTS_DIR / "multi_geom_training" / "l_bracket_horizontal.csv",
    label="L-bracket Horizontal Tip",
    has_header=True,
)
scatter_fig.show()



## Model Architecture & Training Diagnostics

We rebuild the training experiment inline: load the dataset, encode screw + geometry/load features, train a compact surrogate, and compare the results with the production checkpoint stored in `artifacts_multi_geom/`.



In [7]:
metadata_path = PINN_DIR / "artifacts_multi_geom" / "dataset_metadata.json"
stats_path = PINN_DIR / "artifacts_multi_geom" / "norm_stats_multi_geom.npz"
model_path = PINN_DIR / "artifacts_multi_geom" / "pinn_multi_geom.pth"

metadata = json.loads(metadata_path.read_text())
prod_model = SurrogateModel(
    input_dim=metadata["input_dim"],
    hidden_size=metadata["hidden_size"],
)
prod_model.load_state_dict(torch.load(model_path, map_location="cpu"))
prod_model.eval()
param_count = sum(p.numel() for p in prod_model.parameters())

model_summary = pd.DataFrame(
    {
        "num_samples": [metadata["num_samples"]],
        "input_dim": [metadata["input_dim"]],
        "hidden_size": [metadata["hidden_size"]],
        "parameters": [param_count],
        "geometries": [", ".join(metadata["geom_names"])],
        "load_cases": [", ".join(metadata["load_cases"])],
    }
)
display(model_summary)



,num_samples,input_dim,hidden_size,parameters,geometries,load_cases
0,1020,14,96,10849,"l_bracket, ribbed_channel, tapered_plate","combined_tip, horizontal_tip, lateral_shear, u..."


**What this shows:**
- **Model Architecture**: Input dimension (14 = 6 screw coords + 3 geometry one-hot + 5 load case one-hot), hidden size (96 neurons)
- **Parameter Count**: Total trainable weights in the neural network
- **Training Data**: Number of samples used to train this model
- **Geometries & Load Cases**: Lists all scenarios the model was trained on
- **Purpose**: Documents the model capacity and training scope - helps understand what the model can and cannot generalize to

In [8]:
def encode_dataset(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, Dict]:
    geom_names = sorted(df["geometry"].unique())
    load_cases = sorted(df["load_case"].unique())
    geom_map = {name: idx for idx, name in enumerate(geom_names)}
    load_map = {name: idx for idx, name in enumerate(load_cases)}

    geom_one_hot = np.eye(len(geom_names))[df["geometry"].map(geom_map)]
    load_one_hot = np.eye(len(load_cases))[df["load_case"].map(load_map)]
    screw_data = df[SCREW_COLS].values.astype(np.float32)
    X = np.hstack([screw_data, geom_one_hot, load_one_hot]).astype(np.float32)
    y = df["compliance"].values.astype(np.float32).reshape(-1, 1)
    meta = {
        "geom_names": geom_names,
        "load_cases": load_cases,
        "input_dim": X.shape[1],
    }
    return X, y, meta


def train_with_history(X: np.ndarray, y: np.ndarray, *, hidden: int = 96, epochs: int = 400, lr: float = 1e-3):
    X_mean = X.mean(axis=0)
    X_std = X.std(axis=0)
    y_mean = y.mean(axis=0)
    y_std = y.std(axis=0)
    X_std[X_std == 0] = 1.0
    y_std[y_std == 0] = 1.0

    X_norm = (X - X_mean) / X_std
    y_norm = (y - y_mean) / y_std

    X_tensor = torch.tensor(X_norm, dtype=torch.float32)
    y_tensor = torch.tensor(y_norm, dtype=torch.float32)

    model = SurrogateModel(input_dim=X_tensor.shape[1], hidden_size=hidden)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    history = []
    for epoch in range(epochs):
        optimizer.zero_grad()
        pred = model(X_tensor)
        loss = loss_fn(pred, y_tensor)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    stats = {"X_mean": X_mean, "X_std": X_std, "y_mean": y_mean, "y_std": y_std}
    return model, stats, history


X_full, y_full, meta = encode_dataset(multi_df)
sanity_model, sanity_stats, loss_history = train_with_history(X_full, y_full, epochs=250, lr=5e-4)
print(f"Finished sanity training → final normalized MSE {loss_history[-1]:.6f}")

loss_fig = px.line(y=loss_history, title="Inline Training Loss", labels={"index": "Epoch", "value": "MSE"})
loss_fig.show()



Finished sanity training → final normalized MSE 0.011924


**What this shows:**
- **Training Loss Curve**: Mean Squared Error (MSE) decreasing over training epochs
- **Convergence**: Steep initial drop indicates the model is learning quickly, then plateaus as it fine-tunes
- **Normalized MSE**: Values are normalized (0-1 scale) - lower is better
- **Purpose**: 
  - Monitors training progress - should decrease smoothly
  - Helps detect overfitting (loss stops decreasing) or underfitting (loss plateaus too high)
  - Validates that training is working correctly

In [9]:
def denormalize(pred_norm: np.ndarray, stats: dict) -> np.ndarray:
    return pred_norm * stats["y_std"] + stats["y_mean"]


def evaluate_model(model: nn.Module, stats: dict, X: np.ndarray, y: np.ndarray) -> pd.Series:
    X_norm = (X - stats["X_mean"]) / stats["X_std"]
    with torch.no_grad():
        pred_norm = model(torch.tensor(X_norm, dtype=torch.float32)).numpy()
    preds = denormalize(pred_norm, stats)
    mae = np.mean(np.abs(preds - y))
    mape = np.mean(np.abs((preds - y) / y)) * 100
    r2 = 1 - np.sum((preds - y) ** 2) / np.sum((y - y.mean()) ** 2)
    return pd.Series({"MAE (J)": mae, "MAPE (%)": mape, "R^2": r2})

prod_stats_np = np.load(stats_path)
prod_stats = {k: prod_stats_np[k] for k in prod_stats_np.files}
prod_metrics = evaluate_model(prod_model, prod_stats, X_full, y_full)
sanity_metrics = evaluate_model(sanity_model, sanity_stats, X_full, y_full)

comparison = pd.DataFrame(
    {
        "Production": prod_metrics,
        "Inline Sanity": sanity_metrics,
    }
)
display(comparison)

parity_fig = px.scatter(
    x=y_full.flatten(),
    y=denormalize(
        prod_model(torch.tensor((X_full - prod_stats["X_mean"]) / prod_stats["X_std"], dtype=torch.float32)).detach().numpy(),
        prod_stats,
    ).flatten(),
    title="PINN Prediction Parity",
    labels={"x": "FEA Compliance (J)", "y": "PINN Prediction (J)"},
)
parity_fig.add_trace(
    go.Scatter(x=[y_full.min(), y_full.max()], y=[y_full.min(), y_full.max()], mode="lines", name="Ideal")
)
parity_fig.show()



,Production,Inline Sanity
MAE (J),6.157596,28.907059
MAPE (%),55.004837,393.190277
R^2,0.999476,0.988112


**What this shows:**
- **Production vs Inline Model**: Compares the saved production model with a freshly trained "sanity check" model
- **Metrics**:
  - **MAE (Mean Absolute Error)**: Average prediction error in Joules - lower is better
  - **MAPE (Mean Absolute Percentage Error)**: Error as percentage of true value - shows relative accuracy
  - **R² (R-squared)**: Coefficient of determination (0-1) - 1.0 = perfect predictions, 0.0 = no better than mean
- **Purpose**: Validates that the production model performs well and that training is reproducible

## Validation & Generalization Figures

We compare the refreshed multi-geometry surrogate against the legacy single-geometry model using the curated metrics CSV, then explore the generalization samples.



In [10]:
metrics_path = ROOT / "src/experiments/scenario_validation/results/multi_geom_model_metrics.csv"
metrics_df = pd.read_csv(metrics_path)
mae_fig = px.bar(
    metrics_df,
    x="geometry",
    y="multi_mae",
    color="load_case",
    title="Multi-Geometry PINN MAE by Scenario",
    labels={"multi_mae": "MAE (J)"},
)
mae_fig.show()

relative_fig = px.bar(
    metrics_df.melt(
        id_vars=["geometry", "load_case"],
        value_vars=["multi_mape_pct", "legacy_mape_pct"],
        var_name="model",
        value_name="MAPE (%)",
    ),
    x="geometry",
    y="MAPE (%)",
    color="model",
    facet_col="load_case",
    title="MAPE Comparison vs Legacy PINN",
)
relative_fig.update_yaxes(matches=None)
relative_fig.show()



**What this shows:**
- **MAE by Scenario**: Bar chart showing prediction error (in Joules) for each geometry/load case combination
- **Color by Load Case**: Different load cases have different colors
- **Lower is Better**: Smaller bars mean more accurate predictions
- **Purpose**: Identifies which scenarios the model handles well vs. poorly - helps understand generalization limits

In [11]:
generalization_path = RESULTS_DIR / "pinn_generalization_test.csv"
gen_df = pd.read_csv(generalization_path)

# Run the production model on the same screw placements for reproducibility
geom_one_hot = np.eye(len(metadata["geom_names"]))[0]  # assume l_bracket
load_one_hot = np.eye(len(metadata["load_cases"]))[metadata["load_cases"].index("horizontal_tip")]
repeated_geom = np.repeat(geom_one_hot[None, :], len(gen_df), axis=0)
repeated_load = np.repeat(load_one_hot[None, :], len(gen_df), axis=0)
screw_data = gen_df[["s1_x", "s1_y", "s2_x", "s2_y"]].values
if screw_data.shape[1] < metadata["input_dim"] - len(geom_one_hot) - len(load_one_hot):
    padding = metadata["input_dim"] - len(geom_one_hot) - len(load_one_hot) - screw_data.shape[1]
    screw_data = np.pad(screw_data, ((0, 0), (0, padding)), mode="constant")
X_gen = np.hstack([screw_data, repeated_geom, repeated_load])
with torch.no_grad():
    preds_norm = prod_model(torch.tensor((X_gen - prod_stats["X_mean"]) / prod_stats["X_std"], dtype=torch.float32)).numpy()
    preds = denormalize(preds_norm, prod_stats)

gen_fig = px.scatter_3d(
    x=gen_df["s1_x"],
    y=gen_df["s1_y"],
    z=preds.flatten(),
    color=preds.flatten(),
    title="Generalization Samples (PINN Predictions)",
    labels={"x": "s1_x", "y": "s1_y", "z": "Compliance (J)"},
)
gen_fig.show()



**What this shows:**
- **3D Generalization Space**: Screw positions (s1_x, s1_y) vs. predicted compliance (z-axis)
- **Color Gradient**: Red = high compliance (flexible), blue = low compliance (stiff)
- **Surface Shape**: Shows how compliance varies across the design space
- **Purpose**: Visualizes the learned compliance landscape - reveals where the model predicts optimal (low compliance) screw placements

## Reusable PINN Prediction Helper

Expose a small utility so other notebooks (master, MMC) can request compliance estimates without reloading weights manually.



In [12]:
geom_index = {name: idx for idx, name in enumerate(metadata["geom_names"])}
load_index = {name: idx for idx, name in enumerate(metadata["load_cases"])}

def pinn_predict(s1_xy: Tuple[float, float], s2_xy: Tuple[float, float], *, geometry: str = "l_bracket", load_case: str = "horizontal_tip") -> float:
    geom_vec = np.eye(len(metadata["geom_names"]))[geom_index[geometry]]
    load_vec = np.eye(len(metadata["load_cases"]))[load_index[load_case]]
    screw_vec = np.array([*s1_xy, *s2_xy], dtype=np.float32)
    if len(screw_vec) < len(SCREW_COLS):
        screw_vec = np.pad(screw_vec, (0, len(SCREW_COLS) - len(screw_vec)))
    features = np.hstack([screw_vec, geom_vec, load_vec])
    with torch.no_grad():
        pred_norm = prod_model(torch.tensor(((features - prod_stats["X_mean"]) / prod_stats["X_std"]), dtype=torch.float32))
    pred_norm_np = pred_norm.detach().cpu().numpy()
    denormalized = denormalize(pred_norm_np, prod_stats)
    # Extract scalar value: flatten array and take first element
    value = float(np.asarray(denormalized).flatten()[0])
    return value

example_value = pinn_predict((20.0, 30.0), (15.0, 80.0))
print(f"Example compliance prediction: {example_value:.2f} J")



Example compliance prediction: 250.35 J


## Where to Go Next

- Jump back to `master_story.ipynb` for the comparison narrative.  
- Open `mmc_story.ipynb` once it’s ready to cross-check optimization paths against these surrogate predictions.  
- Regenerate this notebook after updating datasets or retraining the PINN so every figure remains in sync.

